# vocalexp tutorial — transform vocal expressivity on a WAV file

This notebook runs the **`vocalexp`** GStreamer plugin on an input WAV file and
lets you play with its parameters. The plugin tracks the pitch of a voice and
rescales the variability of its pitch contour:

- `expressivity = 1.0` → unchanged
- `expressivity = 0.0` → monotone (flattened intonation)
- `expressivity = 2.0` → exaggerated intonation (2× the pitch excursions)

`envelope-preservation` keeps the formants (vocal identity) in place while the
pitch moves.

**Requirements**

- The plugin built in `../build` (`cmake -B build && cmake --build build -j` from the repo root)
- `gst-launch-1.0` from the same GStreamer installation the plugin was built against
- Python: `numpy`, `matplotlib` (`pip install numpy matplotlib`)


## 1. Parameters — edit me

Point `INPUT_WAV` at any WAV file (any rate / channels — the pipeline converts
to mono 48 kHz). Leave it as `None` to auto-generate a demo vibrato tone so the
notebook works out of the box.

In [ ]:
INPUT_WAV = "../media/in/F21a1.wav"   # Pointing to one of the new media files

EXPRESSIVITY = 2.0        # 0.0 .. 4.0   (1.0 = unchanged)
ENVELOPE_PRESERVATION = True

# Advanced (defaults are sensible for voice at 48 kHz)
FRAME_SIZE = 1024         # STFT window, power of two; latency = frame - frame/overlap
OVERLAP_FACTOR = 4        # hop = FRAME_SIZE / OVERLAP_FACTOR
MIN_FREQUENCY = 70.0      # pitch search range (Hz) - narrow it to your voice
MAX_FREQUENCY = 400.0    # for less tracking latency and fewer octave errors

OUTPUT_WAV = "../media/out/out.wav"

## 2. Locate the plugin and gst-launch

In [ ]:
import os, pathlib, shutil, subprocess

REPO_ROOT = pathlib.Path.cwd().resolve()
if not (REPO_ROOT / "CMakeLists.txt").exists():     # running from notebooks/
    REPO_ROOT = REPO_ROOT.parent
BUILD_DIR = REPO_ROOT / "build"

# 1. Check if 'vocalexp' is already available (e.g. in Docker environment)
HAS_PLUGIN = subprocess.run(["gst-inspect-1.0", "vocalexp"], 
                            capture_output=True).returncode == 0

if not HAS_PLUGIN:
    # 2. Not in registry, check for local build
    PLUGIN_SO = BUILD_DIR / "libgstvocalexp.so"
    if not PLUGIN_SO.exists():
        print("Plugin not found in registry or build/ - building locally...")
        subprocess.run(["cmake", "-B", str(BUILD_DIR), "-DCMAKE_BUILD_TYPE=Release"],
                       cwd=REPO_ROOT, check=True)
        subprocess.run(["cmake", "--build", str(BUILD_DIR), "-j"], cwd=REPO_ROOT, check=True)
    
    # Add local build to path
    os.environ["GST_PLUGIN_PATH"] = os.environ.get("GST_PLUGIN_PATH", "") + ":" + str(BUILD_DIR)
    print(f"Using local build at {BUILD_DIR}")
else:
    print("Plugin 'vocalexp' found in GStreamer registry (using Docker/installed version)")

GST_LAUNCH = os.environ.get("GST_LAUNCH") or shutil.which("gst-launch-1.0")
assert GST_LAUNCH, "gst-launch-1.0 not found - install GStreamer or set GST_LAUNCH"

ENV = os.environ.copy()
print("gst-launch:", GST_LAUNCH)

## 3. Input file

If `INPUT_WAV` is `None`, generate a 4 s "singing" tone: 220 Hz with ±15 Hz
vibrato plus a slow melody, harmonics shaped by two vocal formants — enough
structure for the pitch tracker and the envelope preservation to act on.

In [ ]:
import math, struct, wave

import numpy as np

SR = 48000

def write_wav_mono16(path, samples, sr=SR):
    samples = np.clip(samples, -1.0, 1.0)
    with wave.open(str(path), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes((samples * 32000).astype("<i2").tobytes())

if INPUT_WAV is None:
    INPUT_WAV = "in.wav"
    t = np.arange(4 * SR) / SR
    melody = 220 * 2 ** (np.round(2 * np.sin(2 * np.pi * 0.4 * t)) * 2 / 12)
    f0 = melody + 15 * np.sin(2 * np.pi * 5.0 * t)          # vibrato
    phase = 2 * np.pi * np.cumsum(f0) / SR
    x = np.zeros_like(t)
    for h in range(1, 12):                                   # formants at 600/1700 Hz
        fh = np.minimum(h * f0, SR / 2 - 1)
        gain = (np.exp(-0.5 * ((fh - 600) / 250) ** 2) +
                0.5 * np.exp(-0.5 * ((fh - 1700) / 350) ** 2) + 0.05)
        x += gain * np.sin(h * phase) / h
    x *= 0.35 / np.max(np.abs(x))
    write_wav_mono16(INPUT_WAV, x)
    print("generated demo input:", INPUT_WAV)
else:
    print("using input:", INPUT_WAV)

## 4. Run the plugin

The exact pipeline `gst-launch-1.0` runs (the notebook just fills in the
parameters):

```
filesrc ! wavparse ! audioconvert ! audioresample !
  audio/x-raw,format=F32LE,channels=1,rate=48000 !
  vocalexp expressivity=E envelope-preservation=... !
  audioconvert ! wavenc ! filesink
```

In [ ]:
def run_vocalexp(input_wav, output_wav, expressivity, envelope=True,
                 frame_size=1024, overlap=4, fmin=60.0, fmax=1000.0):
    pipeline = [
        GST_LAUNCH, "-q",
        "filesrc", f"location={input_wav}", "!", "wavparse", "!",
        "audioconvert", "!", "audioresample", "!",
        "audio/x-raw,format=F32LE,channels=1,rate=48000", "!",
        "vocalexp",
        f"expressivity={expressivity}",
        f"envelope-preservation={'true' if envelope else 'false'}",
        f"frame-size={frame_size}",
        f"overlap-factor={overlap}",
        f"min-frequency={fmin}",
        f"max-frequency={fmax}",
        "!", "audioconvert", "!", "wavenc", "!", "filesink", f"location={output_wav}",
    ]
    result = subprocess.run(pipeline, env=ENV, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"pipeline failed:\n{result.stderr}")
    return output_wav

run_vocalexp(INPUT_WAV, OUTPUT_WAV, EXPRESSIVITY, ENVELOPE_PRESERVATION,
             FRAME_SIZE, OVERLAP_FACTOR, MIN_FREQUENCY, MAX_FREQUENCY)
print("wrote", OUTPUT_WAV)

## 5. Listen

In [ ]:
from IPython.display import Audio, display

def load_wav(path):
    """Reads PCM16 or float32 WAV, returns (sample_rate, mono float array)."""
    blob = pathlib.Path(path).read_bytes()
    assert blob[:4] == b"RIFF" and blob[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, channels, sr, data = 12, 1, 1, SR, b""
    while pos < len(blob) - 8:
        cid = blob[pos:pos + 4]
        size = struct.unpack("<I", blob[pos + 4:pos + 8])[0]
        body = blob[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            fmt, channels, sr = struct.unpack("<HHI", body[:8])
        elif cid == b"data":
            data = body
        pos += 8 + size + (size & 1)
    if fmt == 3:
        x = np.frombuffer(data, dtype="<f4").astype(np.float64)
    elif fmt == 1:
        x = np.frombuffer(data, dtype="<i2").astype(np.float64) / 32768.0
    else:
        raise ValueError(f"unsupported WAV format tag {fmt}")
    return sr, x.reshape(-1, channels).mean(axis=1)

sr_in, x_in = load_wav(INPUT_WAV)
sr_out, x_out = load_wav(OUTPUT_WAV)

print(f"input  (expressivity = 1.0 by definition)")
display(Audio(x_in, rate=sr_in))
print(f"output (expressivity = {EXPRESSIVITY}, envelope-preservation = {ENVELOPE_PRESERVATION})")
display(Audio(x_out, rate=sr_out))

## 6. See the effect — pitch contours

A simple autocorrelation pitch tracker (analysis only, independent of the
plugin's YIN) plots both contours. With `EXPRESSIVITY > 1` the output contour
swings wider than the input; with `EXPRESSIVITY < 1` it shrinks toward
monotone.

In [ ]:
import matplotlib.pyplot as plt

def pitch_contour(x, sr, fmin=80.0, fmax=600.0, frame=0.04, hop=0.01):
    n, h = int(frame * sr), int(hop * sr)
    lag_min, lag_max = int(sr / fmax), int(sr / fmin)
    times, f0s = [], []
    for start in range(0, len(x) - n - lag_max, h):
        seg = x[start:start + n + lag_max]
        seg = seg - seg.mean()
        if np.sqrt(np.mean(seg[:n] ** 2)) < 1e-3:
            continue
        ref = seg[:n]
        # normalised cross-correlation over the lag range
        corr = np.array([np.dot(ref, seg[lag:lag + n]) /
                         (np.linalg.norm(ref) * np.linalg.norm(seg[lag:lag + n]) + 1e-12)
                         for lag in range(lag_min, lag_max)])
        best = int(np.argmax(corr))
        if corr[best] < 0.7:
            continue
        lag = lag_min + best
        if 0 < best < len(corr) - 1:        # parabolic refinement
            l, c, r = corr[best - 1], corr[best], corr[best + 1]
            if (l - 2 * c + r) != 0:
                lag += 0.5 * (l - r) / (l - 2 * c + r)
        times.append(start / sr)
        f0s.append(sr / lag)
    return np.array(times), np.array(f0s)

t_in, f_in = pitch_contour(x_in, sr_in)
t_out, f_out = pitch_contour(x_out, sr_out)

plt.figure(figsize=(12, 4.5))
plt.plot(t_in, f_in, ".", ms=3, label="input")
plt.plot(t_out, f_out, ".", ms=3, label=f"output (E = {EXPRESSIVITY})")
plt.xlabel("time (s)"); plt.ylabel("f0 (Hz)")
plt.title("Pitch contour before / after vocalexp")
plt.legend(); plt.grid(alpha=0.3); plt.show()

dev_in, dev_out = np.std(f_in), np.std(f_out)
print(f"pitch contour std: input {dev_in:5.2f} Hz -> output {dev_out:5.2f} Hz "
      f"(ratio {dev_out / dev_in:.2f}, expressivity setting {EXPRESSIVITY})")

## 7. Spectrograms

With `envelope-preservation=true` the horizontal bands of spectral energy
(formants) stay at the same heights even where the harmonics move — that is
what keeps the voice sounding like the same person.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, (signal, sr, title) in zip(axes, [(x_in, sr_in, "input"),
                                          (x_out, sr_out, f"output (E = {EXPRESSIVITY})")]):
    ax.specgram(signal, NFFT=1024, Fs=sr, noverlap=768, cmap="magma", vmin=-120)
    ax.set_ylim(0, 4000); ax.set_ylabel("Hz"); ax.set_title(title)
axes[1].set_xlabel("time (s)")
plt.tight_layout(); plt.show()

## 8. Parameter sweep

Render the same input at several expressivity values and compare the contours.
Re-run with your own `INPUT_WAV` and tweak the list below.

In [ ]:
SWEEP = [0.0, 0.5, 1.0, 2.0, 3.0]

plt.figure(figsize=(12, 4.5))
for e in SWEEP:
    out = f"out_E{e}.wav"
    run_vocalexp(INPUT_WAV, out, e, ENVELOPE_PRESERVATION,
                 FRAME_SIZE, OVERLAP_FACTOR, MIN_FREQUENCY, MAX_FREQUENCY)
    sr_e, x_e = load_wav(out)
    t_e, f_e = pitch_contour(x_e, sr_e)
    plt.plot(t_e, f_e, ".", ms=2.5, label=f"E = {e}")
plt.xlabel("time (s)"); plt.ylabel("f0 (Hz)")
plt.title("Pitch contour vs expressivity")
plt.legend(markerscale=3); plt.grid(alpha=0.3); plt.show()

## Notes

- **Latency** — the plugin adds `frame-size − frame-size/overlap-factor`
  samples (16 ms at the defaults @ 48 kHz). Halve `frame-size` for lower
  latency at some quality cost; raise `min-frequency` accordingly so the
  pitch tracker stays reliable (`min-frequency ≳ 2 · rate / frame-size`).
- **Voice range** — narrowing `min-frequency`/`max-frequency` around the
  speaker (e.g. 80–400 Hz for a male voice) reduces tracking latency and
  octave errors.
- **envelope-preservation** — leave it on for natural voices; switching it
  off gives the classic "chipmunk/giant" vocoder colour when pitch moves far.
- For **live microphone** use, see the pipelines in the README and the
  interactive `build/vocalexp-demo` tool.
